In [166]:
import re
import json
import pandas as pd

# Read datasets

In [177]:
def read_dataset(name, split):
    return json.load(open(f'../data/preprocessed/{name}/{name}_{split}.json'))

# Make tasks

In [46]:
system_prompt = """You are a text-to-SPARQL converter for Wikidata. Given a natural language question (QUESTION) and a set of extracted entities (QUESTION ENTITIES) - Wikidata IDs and labels, generate an optimized SPARQL query that retrieves relevant data from Wikidata's query service. Ensure the query is efficient, using appropriate properties, filters, and service clauses where necessary.
Do not include any explanations, comments, or additional text before or after the SPARQL query. Output only the SPARQL query enclosed within triple backticks (```)."""

question_task = lambda question, id2alias, lang='en': f'''QUESTION: {question}\nQUESTION ENTITIES:\n{format_id2alias(id2alias, lang)}'''

In [213]:
# response_format = {
#     "type": "json_schema",
#     "json_schema": {
#         "name": "SPARQL_query",
#         "strict": True,
#         "schema": {
#             "type": "object",
#             "properties": {
#                 "sparql_query": {"type": "string"}
#             },
#             "required": ["sparql_query"],
#             "additionalProperties": False
#         }
#     }


def format_id2alias(id2alias, lang='en'):
    alias_list = []
    for wikidata_id, alias_lang_dict in id2alias.items():
        if alias_lang_dict:
            label = alias_lang_dict.get(lang)
            alias_list.append(f'{wikidata_id}: {label}')
    return '\n'.join(alias_list)

def create_task(item, lang='en'):
    question = item[f'{lang}_question']
    id2alias = item['entities']['question'] if item['entities']['question'] not in (None, {}) else item['entities']['query']

    return question_task(question, id2alias, lang)

def create_request(request_id, task, metadata, model_name, system_prompt=system_prompt):
    return {
        "custom_id": request_id,
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": model_name,
            "store": True,
            "messages": [
                {
                  "role": "system",
                  "content": system_prompt
                },
                {
                  "role": "user",
                  "content": task,
                }
            ],
            "metadata": metadata,
           # "response_format": response_format,
            "seed": 42,      
        }
    }

def item2request(item, dataset_name, model_name='gpt-4', lang='en'):
    request_id = str(item['id'])
    question = item[f'{lang}_question']
    task = create_task(item)
    metadata = {
        "dataset": dataset_name,
        "question": question
    }
    
    return create_request(request_id, task, metadata, model_name=model_name)

def create_jsonl_file(requests_list, file_path):
    with open(file_path, "w", encoding="utf-8") as f:
        for request in requests_list:
            f.write(json.dumps(request, ensure_ascii=False) + "\n")

    print(f"JSONL file created successfully at {file_path}")


def extract_sparql(sparql_string):
    match = re.search(r'```(.*?)```', sparql_string, re.DOTALL)
    if match:
        return match.group(1).strip()
    else:
        raise ValueError("No SPARQL query found within triple backticks.")

# Test

In [106]:
from openai import OpenAI

API_KEY = ''
client = OpenAI(api_key=API_KEY)

model_name = 'gpt-4'
lang = 'en'

# RuBQ

In [107]:
dataset_name = 'rubq'

dataset = read_dataset(dataset_name, 'test')['dataset']
assert len({d['id'] for d in dataset}) == len(dataset), "Duplicate IDs found!"

requests_list = [item2request(item, dataset_name, model_name=model_name, lang=lang) for item in dataset]
batch_filepath = f"../artefacts/gpt_as_kgqa/batches/{dataset_name}_batch.jsonl"

create_jsonl_file(requests_list, batch_filepath)

batch_input_file = client.files.create(
    file=open(batch_filepath, "rb"),
    purpose="batch"
)

batch = client.batches.create(
    input_file_id=batch_input_file.id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
    metadata={
        "dataset": dataset_name
    }
)

JSONL file created successfully at ../artefacts/gpt_as_kgqa/batches/rubq_batch.jsonl


In [189]:
file_id = 'file-LrYj9Gj7QtP4nFo8ZSH1HZ'
batch_id = 'batch_679e3fe3ce8c819092fba6f13412c6fb'

batch = client.batches.retrieve(batch_id)
print(batch)

Batch(id='batch_679e3fe3ce8c819092fba6f13412c6fb', completion_window='24h', created_at=1738424291, endpoint='/v1/chat/completions', input_file_id='file-LrYj9Gj7QtP4nFo8ZSH1HZ', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1738425104, error_file_id=None, errors=None, expired_at=None, expires_at=1738510691, failed_at=None, finalizing_at=1738425026, in_progress_at=1738424293, metadata={'dataset': 'rubq'}, output_file_id='file-NXRDckATg2SRMKRH1K5NpS', request_counts=BatchRequestCounts(completed=480, failed=0, total=480))


In [190]:
batch = client.batches.retrieve(batch_id)
output_file_id = batch.output_file_id

file_response = client.files.content(output_file_id)
responses = [json.loads(line) for line in file_response.text.splitlines()]

gpt_predicted_sparqls = {response['custom_id']: extract_sparql(response['response']['body']['choices'][0]['message']['content']) for response in responses}
id2question = {str(item['id']): item['en_question'] for item in dataset}
id2sparql = {str(item['id']): item['query'] for item in dataset}
assert set(gpt_predicted_sparqls.keys()) == set(id2question.keys())

prediction = pd.DataFrame({
    'question': id2question,
    'sparql': id2sparql,
    'gpt_sparql': gpt_predicted_sparqls
})

prediction.to_csv(f'../artefacts/gpt_as_kgqa/results/{dataset_name}.csv')

In [191]:
prediction.head()

,question,sparql,gpt_sparql
4,Which country does the famous Easter island be...,select ?answer where { wd:Q14452 wdt:P17 ?answ...,SELECT ?country WHERE {\n wd:Q14452 wdt:P17 ?...
7,Which music group is Mick Jagger's name inextr...,select ?answer where { wd:Q128121 wdt:P361 ?an...,SELECT ?group WHERE {\n wd:Q128121 wdt:P463 ?...
14,Where is the Summer garden?,select ?answer where { wd:Q1229234 wdt:P131 ?a...,SELECT ?location WHERE {\n wd:Q1229234 wdt:P2...
22,Which city is the capital of Turkmenistan?,select ?answer where { wd:Q874 wdt:P36 ?answer },SELECT ?city WHERE {\n wd:Q874 wdt:P36 ?city....
25,In which city was the first Russian revolution...,select ?answer where { wd:Q2533402 wdt:P159 ?a...,SELECT ?cityLabel\nWHERE\n{\n wd:Q2533402 wdt...


# Qald

In [170]:
dataset_name = 'qald'

dataset = read_dataset(dataset_name, 'test')['dataset']
assert len({d['id'] for d in dataset}) == len(dataset), "Duplicate IDs found!"

requests_list = [item2request(item, dataset_name, model_name=model_name, lang=lang) for item in dataset]
batch_filepath = f"../artefacts/gpt_as_kgqa/batches/{dataset_name}_batch.jsonl"

create_jsonl_file(requests_list, batch_filepath)

batch_input_file = client.files.create(
    file=open(batch_filepath, "rb"),
    purpose="batch"
)

batch = client.batches.create(
    input_file_id=batch_input_file.id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
    metadata={
        "dataset": dataset_name
    }
)

JSONL file created successfully at ../artefacts/gpt_as_kgqa/batches/qald_batch.jsonl


In [195]:
file_id = 'file-YWbdCcihYNJptSLTULQgUH'
batch_id = 'batch_679fc0a4fea08190831508495658b5a3'

batch = client.batches.retrieve(batch_id)
print(batch)

Batch(id='batch_679fc0a4fea08190831508495658b5a3', completion_window='24h', created_at=1738522789, endpoint='/v1/chat/completions', input_file_id='file-YWbdCcihYNJptSLTULQgUH', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1738523619, error_file_id=None, errors=None, expired_at=None, expires_at=1738609189, failed_at=None, finalizing_at=1738523588, in_progress_at=1738522791, metadata={'dataset': 'qald'}, output_file_id='file-9tHsSTHJnaTGgt4TCoZrTW', request_counts=BatchRequestCounts(completed=386, failed=0, total=386))


In [197]:
batch = client.batches.retrieve(batch_id)
output_file_id = batch.output_file_id

file_response = client.files.content(output_file_id)
responses = [json.loads(line) for line in file_response.text.splitlines()]

gpt_predicted_sparqls = {response['custom_id']: extract_sparql(response['response']['body']['choices'][0]['message']['content']) for response in responses}
id2question = {str(item['id']): item['en_question'] for item in dataset}
id2sparql = {str(item['id']): item['query'] for item in dataset}
assert set(gpt_predicted_sparqls.keys()) == set(id2question.keys())

prediction = pd.DataFrame({
    'question': id2question,
    'sparql': id2sparql,
    'gpt_sparql': gpt_predicted_sparqls
})

prediction.to_csv(f'../artefacts/gpt_as_kgqa/results/{dataset_name}.csv')

In [198]:
prediction.head()

,question,sparql,gpt_sparql
0,After whom is the Riemannian geometry named?,select distinct ?result where { wd:Q761383 wdt...,SELECT ?answer WHERE {\n wd:Q761383 wdt:P138 ...
1,Which animal participated in a military operat...,select distinct ?result where { ?result wdt:P3...,SELECT ?animal WHERE {\n ?operation wdt:P31 w...
2,"among the characters in the witcher, who has t...",select distinct ?result where { wd:Q11835640 w...,SELECT ?character WHERE {\n ?character wdt:P6...
3,"among the founders of tencent company, who has...",select distinct ?result where { wd:Q860580 wdt...,SELECT ?founder ?founderLabel WHERE {\n ?comp...
4,among the other representative work of the aut...,select distinct ?result where { wd:Q696071 wdt...,SELECT ?work ?workLabel WHERE {\n ?author wdt...


### LcQuad

In [204]:
dataset_name = 'lcquad_2.0'

dataset = read_dataset(dataset_name, 'test')['dataset']
assert len({d['id'] for d in dataset}) == len(dataset), "Duplicate IDs found!"

requests_list = [item2request(item, dataset_name, model_name=model_name, lang=lang) for item in dataset]
batch_filepath = f"../artefacts/gpt_as_kgqa/batches/{dataset_name}_batch.jsonl"

create_jsonl_file(requests_list, batch_filepath)

batch_input_file = client.files.create(
    file=open(batch_filepath, "rb"),
    purpose="batch"
)

batch = client.batches.create(
    input_file_id=batch_input_file.id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
    metadata={
        "dataset": dataset_name
    }
)

JSONL file created successfully at ../artefacts/gpt_as_kgqa/batches/lcquad_2.0_batch.jsonl


In [209]:
file_id = 'file-D8u3ShB9JqVYhebvc3WNtt'
batch_id = 'batch_679fd07ef3008190bca23df216156df9'

batch = client.batches.retrieve(batch_id)
print(batch)

Batch(id='batch_679fd07ef3008190bca23df216156df9', completion_window='24h', created_at=1738526847, endpoint='/v1/chat/completions', input_file_id='file-D8u3ShB9JqVYhebvc3WNtt', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1738529182, error_file_id=None, errors=None, expired_at=None, expires_at=1738613247, failed_at=None, finalizing_at=1738528830, in_progress_at=1738526849, metadata={'dataset': 'lcquad_2.0'}, output_file_id='file-LuN8WN1esJN8HTgkeixPW2', request_counts=BatchRequestCounts(completed=4541, failed=0, total=4541))


In [214]:
batch = client.batches.retrieve(batch_id)
output_file_id = batch.output_file_id

file_response = client.files.content(output_file_id)
responses = [json.loads(line) for line in file_response.text.splitlines()]

gpt_predicted_sparqls = {response['custom_id']: extract_sparql(response['response']['body']['choices'][0]['message']['content']) for response in responses}
id2question = {str(item['id']): item['en_question'] for item in dataset}
id2sparql = {str(item['id']): item['query'] for item in dataset}
assert set(gpt_predicted_sparqls.keys()) == set(id2question.keys())

prediction = pd.DataFrame({
    'question': id2question,
    'sparql': id2sparql,
    'gpt_sparql': gpt_predicted_sparqls
})

prediction.to_csv(f'../artefacts/gpt_as_kgqa/results/{dataset_name}.csv')

There are no entities provided in the question, and "azure" is usually referred to as a color not an entity that could have an eye color. Thus, the question in its current state does not allow writing a meaningful SPARQL query.


In [ ]:
prediction.head()

### PAT

In [248]:
dataset_name = 'pat'

dataset = read_dataset(dataset_name, 'test')['dataset']
assert len({d['id'] for d in dataset}) == len(dataset), "Duplicate IDs found!"

requests_list = [item2request(item, dataset_name, model_name=model_name, lang=lang) for item in dataset]
batch_filepath = f"../artefacts/gpt_as_kgqa/batches/{dataset_name}_batch.jsonl"

create_jsonl_file(requests_list, batch_filepath)

batch_input_file = client.files.create(
    file=open(batch_filepath, "rb"),
    purpose="batch"
)

batch = client.batches.create(
    input_file_id=batch_input_file.id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
    metadata={
        "dataset": dataset_name
    }
)

JSONL file created successfully at ../artefacts/gpt_as_kgqa/batches/pat_batch.jsonl


In [254]:
file_id = 'file-LvjHZSG7eW7SocTc9fKsDW'
batch_id = 'batch_67a1f7d5c8ac819092e171c6135726ef'

batch = client.batches.retrieve(batch_id)
print(batch)

Batch(id='batch_67a1f7d5c8ac819092e171c6135726ef', completion_window='24h', created_at=1738667989, endpoint='/v1/chat/completions', input_file_id='file-LvjHZSG7eW7SocTc9fKsDW', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1738669184, error_file_id=None, errors=None, expired_at=None, expires_at=1738754389, failed_at=None, finalizing_at=1738669092, in_progress_at=1738667991, metadata={'dataset': 'pat'}, output_file_id='file-8euG4sXdhaG2Exk1pj44xG', request_counts=BatchRequestCounts(completed=1199, failed=0, total=1199))


In [255]:
batch = client.batches.retrieve(batch_id)
output_file_id = batch.output_file_id

file_response = client.files.content(output_file_id)
responses = [json.loads(line) for line in file_response.text.splitlines()]

gpt_predicted_sparqls = {response['custom_id']: extract_sparql(response['response']['body']['choices'][0]['message']['content']) for response in responses}
id2question = {str(item['id']): item['en_question'] for item in dataset}
id2sparql = {str(item['id']): item['query'] for item in dataset}
assert set(gpt_predicted_sparqls.keys()) == set(id2question.keys())

prediction = pd.DataFrame({
    'question': id2question,
    'sparql': id2sparql,
    'gpt_sparql': gpt_predicted_sparqls
})

prediction.to_csv(f'../artefacts/gpt_as_kgqa/results/{dataset_name}.csv')

In [256]:
prediction.head()

,question,sparql,gpt_sparql
2308,Who was the previous head coach of the team Ar...,\n SELECT ?item ?itemLabel ?starttime ?endt...,SELECT ?coach ?coachLabel WHERE {\n ?coach ps...
1820,Who is the head of the government of Kertemind...,\n SELECT ?item ?itemLabel (YEAR(?starttime...,SELECT ?head_of_government WHERE {\n wd:Q6124...
2407,Who was the previous head of the government of...,\n SELECT ?item ?itemLabel ?starttime ?endt...,SELECT ?person ?personLabel WHERE {\n ?person...
2417,Who was the previous head of the government of...,\n SELECT ?item ?itemLabel ?starttime ?endt...,SELECT ?previous_head_name WHERE {\n ?previou...
1928,Who is the head of the government of Odense Mu...,\n SELECT ?item ?itemLabel (YEAR(?starttime...,SELECT ?head LABEL WHERE {\n wd:Q21146 wdt:P6...
